In [2]:
%pip install --upgrade spiceypy

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
"""
Utility functions using spiceypy to compute the position of `target` relative to `observer`
at a specified time in a requested reference frame.

Usage:
  - Install spiceypy: pip install spiceypy
  - Provide kernel file paths to `init_spice(kernels)` (you will fill paths to SPICE kernels).
  - Call `position_of_body_relative_to(body_target, body_obs, time, frame='J2000', abcorr='NONE')`.

The functions perform no automatic kernel downloads — you must provide correct kernels
(e.g. leapseconds, planetary ephemeris, spacecraft kernel, etc).
"""

from typing import Iterable, Tuple, Union, Optional
import datetime

import spiceypy as spice


def init_spice(kernel_paths: Iterable[str]) -> None:
    """
    Initialize SPICE by loading the provided kernel files.

    kernel_paths: iterable of file paths (strings) to SPICE kernels (.tm, .bsp, .tls, .tpc, ...)
                  Example order: ['naif0012.tls', 'de440.bsp', 'my_spacecraft.bsp', 'my_fk.tf']
    This function calls spice.kclear() first to ensure a clean kernel pool.
    """
    # clear any previously loaded kernels
    spice.kclear()
    for p in kernel_paths:
        spice.furnsh(p)


def _to_et(time_input: Union[str, datetime.datetime, float, int]) -> float:
    """
    Convert an input time to SPICE ET (ephemeris seconds past J2000).
    Accepts:
      - ISO-like time string (e.g. '2025-03-12T09:05:00.355' or '2025-03-12 09:05:00')
      - datetime.datetime (naive treated as UTC; aware converted to UTC)
      - numeric (assumed already ET seconds) -> returned unchanged as float
    """

    if isinstance(time_input, (float, int)):
        return float(time_input)

    if isinstance(time_input, datetime.datetime):
        dt = time_input
        if dt.tzinfo is None:
            # assume UTC for naive datetime
            ts = dt.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3]  # keep milliseconds
        else:
            dt_utc = dt.astimezone(datetime.timezone.utc)
            ts = dt_utc.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3]
        return spice.str2et(ts)

    if isinstance(time_input, str):
        # pass through to spice.str2et which accepts many ISO formats
        return spice.str2et(time_input)

    raise TypeError("Unsupported time_input type. Use str, datetime, or numeric ET.")


def position_of_body_relative_to(
    target: str,
    observer: str,
    time: Union[str, datetime.datetime, float, int],
    frame: str = "J2000",
    abcorr: str = "NONE",
) -> Tuple[Tuple[float, float, float], float]:
    """
    Compute the geometric position vector of `target` relative to `observer` at `time`
    expressed in `frame`. Returns (position_vector, light_time).

    Parameters:
      - target: name or NAIF id of the target body (e.g. 'MARS', 'EARTH', 'HERA')
      - observer: name or NAIF id of the observer body (e.g. 'HERA' or 'EARTH')
      - time: time string / datetime / ET seconds
      - frame: reference frame string (default 'J2000')
      - abcorr: aberration correction (e.g. 'NONE', 'LT+S', ...)

    Returns:
      - position vector (x, y, z) in km
      - light-time in seconds
    """
    et = _to_et(time)
    # spice.spkpos returns tuple (position_vector, light_time)
    pos, lt = spice.spkpos(target, et, frame, abcorr, observer)
    # pos is a list-like of 3 floats (km)
    return (float(pos[0]), float(pos[1]), float(pos[2])), float(lt)


# Example (uncomment and fill kernel paths before running):
# if __name__ == "__main__":
#     kernels = [
#         "/path/to/naif0012.tls",   # leap seconds
#         "/path/to/de440.bsp",      # planetary ephemeris
#         "/path/to/your_spacecraft.bsp",  # spacecraft ephemeris if needed
#         "/path/to/your_frames.tf", # frame kernels if needed
#     ]
#     init_spice(kernels)
#     pos, lt = position_of_body_relative_to("MARS", "EARTH", "2025-03-12T09:05:00.355", frame="J2000", abcorr="NONE")
#     print("Position (km):", pos, "Light time (s):", lt)

In [ ]:
import os
import numpy as np

os.chdir(r"C:\Users\haral\Desktop\pro3d\spice2\kernels\mk")
kernels = ["C:\\Users\\haral\\Desktop\\pro3d\\spice2\\kernels\\mk\\hera_ops.tm"]
init_spice(kernels)
print(spice.bodn2c("HERA"))
time = "2025-03-12T12:17:01.011Z"
time2 = spice.scs2e(spice.bodn2c("HERA"), "1/0013496591.01370")
pos1, lt1 = position_of_body_relative_to("EARTH", "HERA", time2, frame="J2000", abcorr="NONE")
pos2, lt2 = position_of_body_relative_to("EARTH", "HERA", time, frame="ECLIPJ2000", abcorr="NONE")
pos3, lt3 = position_of_body_relative_to("MARS BARYCENTER", "HERA", time, frame="J2000", abcorr="NONE")
print("Position (km):", pos1, "Light time (s):", lt1)

# AF1_0CRS8F_250312T121701_1B.mbi.json

         #  "EARTPOSX": {
         #      "value": 44786569.872,
         #      "comment": "Earth position vector X [km]"
         #  },
         #  "EARTPOSY": {
         #      "value": -121731385.628,
         #      "comment": "Earth position vector Y [km]"
         #  },
         #  "EARTPOSZ": {
         #      "value": -61491698.042,
         #      "comment": "Earth position vector Z [km]"
         #  },
         #   "TARGET": {
         #       "value": "MARS BARYCENTER",
         #       "comment": ""
         #   },
         #   "TRG_POSX": {
         #       "value": -14235.135,
         #       "comment": "Target position vector X [km]"
         #   },
         #   "TRG_POSY": {
         #       "value": 7038.049,
         #       "comment": "Target position vector Y [km]"
         #   },
         #   "TRG_POSZ": {
         #       "value": 12671.323,
         #       "comment": "Target position vector Z [km]"
         #   },

p = (44786569.872, -121731385.628, -61491698.042)
pos_vec = np.array(pos1)
ref_vec = np.array(p)
diff = pos_vec - ref_vec
print("Difference vector:", diff)
print("Norm of difference (km):", np.linalg.norm(diff))

p2 = (-14235.135, 7038.049, 12671.323)
pos_vec2 = np.array(pos3)
ref_vec2 = np.array(p2)
diff = pos_vec2 - ref_vec2
print("Difference vector:", diff)
print("Norm of difference (km):", np.linalg.norm(diff))

SpiceKERNELVARNOTFOUND: 
================================================================================

Toolkit version: CSPICE_N0067

SPICE(KERNELVARNOTFOUND) --
The Variable Was not Found in the Kernel Pool.
SCLK01_N_FIELDS_150 not found. Did you load the SCLK kernel?

scs2e_c --> SCS2E --> SCENCD --> SCPART --> SCPR01 --> ZZSCUP01 --> ZZSCAD01 --> SCLI01

================================================================================